# CARI-7A and NMDB comparison (archived research code)

This is the source notebook with large embedded execution outputs removed for a lightweight public portfolio. The archived original contained historical matched-station statistics, but the data and CARI .ANS input are not included; this source is not a fully runnable reproduction by itself.


# CARI-7A Modelled Radiation vs NM Station Measurements

## Summary

This notebook evaluates how well the CARI-7A radiation transport model predicts ground-level cosmic ray neutron flux measured by neutron monitor (NM) stations from the Neutron Monitor Database (NMDB).

**What was done:**
- CARI-7A (FAA, v4.2.0) was run for 59 NMDB stations over the period 2000-01-01 to 2026-01-20 (daily average, H0), computing secondary neutron particle flux (D1, P1) at a standardised altitude of 10,000 m (32,808 ft).
- Observed NM count rates (`space_value`) from the corresponding station CSV files were merged with the modelled output by station and date.
- For each station, a linear regression was performed between modelled dose rate and observed count rate, and RÂ² was computed as a measure of association.

**What RÂ² means here:**
RÂ² measures the proportion of variance in the observed NM count rate that is explained by the CARI-7A modelled flux. A value close to 1 indicates strong agreement; a value near 0 indicates little explanatory power.

**Note:** The two datasets operate at different altitudes â CARI-7A was run at 10,000 m (cruising altitude), while NM stations measure at ground level. The comparison therefore tests whether the temporal variability driven by solar modulation is consistent between the two, rather than absolute magnitude.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scipy import stats
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Paths relative to scripts/ folder where this notebook lives
ANS_FILE   = '../dataset/inputs/s7/test.ANS'
CSV_FOLDER = '../dataset/outcomes/from_s6'
OUTPUT_DIR = '../dataset/outcomes/from_s7'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths set. Output folder ready:', OUTPUT_DIR)

## Step 1: Parse test.ANS

In [ ]:
def parse_ans(filepath):
    """
    Parse CARI-7A .ANS output file.
    Data line field order (comma-separated, 0-indexed):
      0:LAT, 1:LON, 2:ALT, 3:F, 4:DATE, 5:HR, 6:VCR, 7:PARTICLE, 8:DOSE_RATE, ...
    Station name is read from comment lines containing 'Station:'.
    """
    records = []
    current_station = None

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()

            # Station comment line e.g. 'C, Station: INVK'
            if 'Station:' in line:
                current_station = line.split('Station:')[1].strip()
                continue

            # Skip header, empty, and other comment lines
            if not line or line.startswith('C') or line.startswith('LAT'):
                continue

            parts = [p.strip() for p in line.split(',')]
            if len(parts) < 9:
                continue
            try:
                date_str  = parts[4]         # e.g. 2000/01/01
                dose_rate = float(parts[8])  # e.g. 3.4461E+00
                date      = pd.to_datetime(date_str, format='%Y/%m/%d')
                records.append({
                    'station':   current_station,
                    'date':      date,
                    'dose_rate': dose_rate
                })
            except:
                continue

    df = pd.DataFrame(records)
    print(f'ANS parsed: {len(df)} rows, {df["station"].nunique()} stations')
    return df

df_ans = parse_ans(ANS_FILE)
df_ans.head()

## Step 2: Load station CSV files

In [ ]:
def load_station_csvs(folder):
    """
    Load all station_XXXX_weather_space_merged.csv files.
    Aggregate two periods per day (h00_12, h13_23) -> daily mean of space_value.
    """
    files = glob.glob(os.path.join(folder, 'station_*_weather_space_merged.csv'))
    frames = []
    for fp in files:
        df = pd.read_csv(fp)
        df['date'] = pd.to_datetime(df['date'])
        df['space_value'] = pd.to_numeric(df['space_value'], errors='coerce')
        daily = df.groupby(['station', 'date'])['space_value'].mean().reset_index()
        frames.append(daily)

    result = pd.concat(frames, ignore_index=True)
    print(f'Station CSVs loaded: {len(result)} rows, {result["station"].nunique()} stations')
    return result

df_obs = load_station_csvs(CSV_FOLDER)
df_obs.head()

## Step 3: Merge ANS and observations

In [ ]:
df_merged = pd.merge(df_ans, df_obs, on=['station', 'date'], how='inner')
df_merged = df_merged.dropna(subset=['dose_rate', 'space_value'])

print(f'Merged: {len(df_merged)} rows, {df_merged["station"].nunique()} stations')
df_merged.head()

## Step 4: Compute RÂ² per station

In [ ]:
results = []

for station, grp in df_merged.groupby('station'):
    if len(grp) < 10:
        continue
    x = grp['dose_rate'].values
    y = grp['space_value'].values
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    results.append({
        'station':   station,
        'n':         len(grp),
        'R2':        r_value**2,
        'r':         r_value,
        'p_value':   p_value,
        'slope':     slope,
        'intercept': intercept
    })

df_r2 = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print(df_r2.to_string(index=False))

## Step 5: RÂ² bar chart across all stations

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['steelblue' if r >= 0.5 else 'salmon' for r in df_r2['R2']]
ax.bar(df_r2['station'], df_r2['R2'], color=colors)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='RÂ²=0.5')
ax.set_xlabel('Station')
ax.set_ylabel('RÂ²')
ax.set_title('RÂ² between CARI-7A Modelled Neutron Flux and NM Observations')
ax.set_xticklabels(df_r2['station'], rotation=90, fontsize=7)
ax.legend()
plt.tight_layout()

out_bar = os.path.join(OUTPUT_DIR, 'r2_by_station.png')
plt.savefig(out_bar, dpi=150)
plt.close()

# Display inline
display(Image(filename=out_bar))

## Step 6: Scatter plots for top 6 stations by RÂ²

In [ ]:
top6 = df_r2.head(6)['station'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, station in enumerate(top6):
    grp  = df_merged[df_merged['station'] == station]
    info = df_r2[df_r2['station'] == station].iloc[0]
    ax   = axes[i]

    ax.scatter(grp['dose_rate'], grp['space_value'], s=5, alpha=0.4, color='steelblue')

    x_line = np.linspace(grp['dose_rate'].min(), grp['dose_rate'].max(), 100)
    y_line = info['slope'] * x_line + info['intercept']
    ax.plot(x_line, y_line, color='red', linewidth=1.2)

    ax.set_title(f"{station}  RÂ²={info['R2']:.3f}  n={info['n']}")
    ax.set_xlabel('CARI-7A Dose Rate (particles/sq.cm/sec)')
    ax.set_ylabel('NM Count Rate')

plt.tight_layout()

out_scatter = os.path.join(OUTPUT_DIR, 'scatter_top6.png')
plt.savefig(out_scatter, dpi=150)
plt.close()

# Display inline
display(Image(filename=out_scatter))

## Step 7: Export RÂ² table

In [ ]:
out_csv = os.path.join(OUTPUT_DIR, 'r2_results.csv')
df_r2.to_csv(out_csv, index=False)
print(f'Saved to {out_csv}')
display(df_r2)

## Results Summary

The analysis compared CARI-7A modelled secondary neutron flux with observed neutron monitor count rates across all available NMDB stations over the period 2000â2026.

**Key findings:**

- Stations with high RÂ² (> 0.5, shown in blue in the bar chart) demonstrate that the temporal variation in CARI-7A modelled flux closely tracks the observed NM count rate. This is consistent with both being driven by the same underlying physical process: solar modulation of galactic cosmic rays (GCR).
- Stations with low RÂ² (< 0.5, shown in red) may reflect data quality issues (sparse observations, gaps), local shielding effects, or differences in the geomagnetic cutoff rigidity between the station location and the model input.
- The scatter plots for the top 6 stations show a clear positive linear relationship, with the red regression line capturing the overall trend well.

**Interpretation:**

Since CARI-7A was run at 10,000 m altitude while NM stations measure at ground level, the comparison is not expected to match in absolute units. However, the solar cycle modulation signal (an ~11-year cycle visible in both datasets) provides a strong common temporal driver. High RÂ² values confirm that CARI-7A successfully captures this modulation, supporting its use as a proxy for cosmic radiation variability.

**Output files saved to** `../dataset/outcomes/from_s7/`:
- `r2_results.csv` â RÂ², r, p-value, slope, intercept and sample size per station
- `r2_by_station.png` â bar chart of RÂ² across all stations
- `scatter_top6.png` â scatter plots with regression lines for the 6 highest-RÂ² stations